# TFIDF - SPAM ASSASIN

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

# Load preprocessed data
print("Loading preprocessed data...")
data = pd.read_csv('../DATASETS/SA_preprocessed.csv')

print(f"Dataset shape: {data.shape}")
print(f"Target distribution:\n{data['label'].value_counts()}")

# Prepare data
X = data['processed_text'].fillna('')
y = data['label'].values

print(f"\nTotal samples: {len(X)}")
print(f"Spam: {sum(y)}, Ham: {len(y) - sum(y)}")

Loading preprocessed data...
Dataset shape: (6045, 4)
Target distribution:
label
0    4149
1    1896
Name: count, dtype: int64

Total samples: 6045
Spam: 1896, Ham: 4149


In [2]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


Training samples: 4836
Testing samples: 1209


In [3]:
print(X_train)

5643    cost effective mlm bizop phone screened leads ...
2113    bush orders sharon to obey un resolutions so u...
2918    re ilug mirror a website you could try httrack...
4146    neat net tricks standard issue number december...
2597    ilug optimising for pentiummmx i have been pla...
                              ...                        
3984    save an extra moneylimit number of compaq s po...
2553    ilug social re ilug linux the film moved from ...
4818    press release the business opportunity allianc...
2585    ilug samba login question i have a situation w...
5426    adv direct email blaster email addresses extra...
Name: processed_text, Length: 4836, dtype: object


In [4]:
tfidf = TfidfVectorizer(
    max_features=10000, # top 10000 words
    ngram_range=(1, 2), # uni and bigrams
    stop_words='english', # remove stop words
)

# Fit and transform
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

TF-IDF features: 10000


In [5]:
print(X_test_tfidf[1])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 28 stored elements and shape (1, 10000)>
  Coords	Values
  (0, 1594)	0.15133228280149197
  (0, 2132)	0.1840877539261862
  (0, 2491)	0.07355827032537328
  (0, 2512)	0.09781936190425644
  (0, 3706)	0.15987912207666852
  (0, 4509)	0.10985310977602657
  (0, 4746)	0.1175691437200736
  (0, 4757)	0.217650004222551
  (0, 4763)	0.15082170911789214
  (0, 4777)	0.14741695358929344
  (0, 4979)	0.09263197086072056
  (0, 4982)	0.09670073378355669
  (0, 5681)	0.0985556927930893
  (0, 5907)	0.06139731408616934
  (0, 6066)	0.23839085699395018
  (0, 6724)	0.17350485599156143
  (0, 7262)	0.1810386020781222
  (0, 7264)	0.21025596257926604
  (0, 7441)	0.3515715943943042
  (0, 7626)	0.2939061239910814
  (0, 7630)	0.30215238633117764
  (0, 8819)	0.14036793974946446
  (0, 9360)	0.1695928772497177
  (0, 9392)	0.23044557038795002
  (0, 9401)	0.18184773330912715
  (0, 9472)	0.22868050653520966
  (0, 9637)	0.17895660353257628
  (0, 9638)	0.263361286776

In [6]:
feature_names = tfidf.get_feature_names_out()
tfidf_matrix = tfidf.transform(X_test)

spam_docs = tfidf_matrix[y_test == 1]

# average TF-IDF across spam docs
mean_tfidf = spam_docs.mean(axis=0).A1

top_idx = np.argsort(mean_tfidf)[::-1]

# Print the words with tfidf that correspond to spam
for i in top_idx[:20]:
    print(feature_names[i], mean_tfidf[i])

number 0.1388995384071436
hyperlink 0.08010833264685244
number number 0.06374447318599613
moneylimit 0.046924862064543084
click 0.04267795566195182
moneylimit number 0.040994214563592056
free 0.03371036083823851
email 0.032309445180700204
hyperlink click 0.027627101226497287
urladdr 0.026381732793047084
credit 0.023445444311785243
hyperlink hyperlink 0.023444864745641985
removed 0.02080344204350244
business 0.01902143749659204
receive 0.018320073931842314
list 0.018210906500065564
money 0.01788497828104812
information 0.017743773149969957
mail 0.016419244600690698
ufffd 0.016123512145174407


In [7]:
print("TF-IDF + Naive Bayes")
nb_clf = MultinomialNB(alpha=0.1)
nb_clf.fit(X_train_tfidf, y_train)
nb_pred = nb_clf.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, nb_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, nb_pred):.4f}")

TF-IDF + Naive Bayes
Accuracy: 0.9752
F1 Score: 0.9602


In [8]:
from sklearn.metrics import classification_report

print(classification_report(y_test, nb_pred, target_names=['Ham', 'Spam']))

              precision    recall  f1-score   support

         Ham       0.98      0.98      0.98       830
        Spam       0.97      0.96      0.96       379

    accuracy                           0.98      1209
   macro avg       0.97      0.97      0.97      1209
weighted avg       0.98      0.98      0.98      1209



In [9]:
print("TF-IDF + SVM")
svm_clf = SVC(kernel='linear', C=1.0, random_state=42)
svm_clf.fit(X_train_tfidf, y_train)
svm_pred = svm_clf.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, svm_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, svm_pred):.4f}")

TF-IDF + SVM
Accuracy: 0.9719
F1 Score: 0.9544


In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_test, svm_pred, target_names=['Ham', 'Spam']))

              precision    recall  f1-score   support

         Ham       0.97      0.99      0.98       830
        Spam       0.97      0.94      0.95       379

    accuracy                           0.97      1209
   macro avg       0.97      0.96      0.97      1209
weighted avg       0.97      0.97      0.97      1209



In [ ]:
misclassified = [
    (text, true, pred)
    for text, true, pred in zip(X_test, y_test, nb_pred)
    if true != pred
]

print(f"Total misclassified samples: {len(misclassified)}\n")

for idx, (text, true, pred) in enumer ate(misclassified, 1):
    print(f"Sample {idx}")
    print(f"True label:      {'Spam' if true == 1 else 'Ham'}")
    print(f"Predicted label: {'Spam' if pred == 1 else 'Ham'}")
    print(f"Text: {text}\n")

Total misclassified samples: 30

Sample 1
True label:      Spam
Predicted label: Ham
Text: amazing trade computer deals please take time to look at our newly designed website www microfinder com we have a full range of notebooks desktops and accessories displayed on the site if you are looking for trade prices feel free to subscribe to the trade section here are some of our latest specials notebooks number x famous brand a piii mhz mb ram gb hdd number number tft screen k modem cd rom fdd usb number number vat number number number x dell cpx h gt piii mhz mb gb hdd number number tft cdrom fdd usb number number vat number number vat dell cdrw available number extra number x ibm thinkpad number xd p mmx mb gb number number tft cdrom fdd usb new battery number number vat number number vat number x digital hi note vp number p mmx mb gb hdd number number tft cdrom fdd usb free carry case number number vat number number vat number x low end notebooks p mb number gb tft and dual scan fdd from

In [55]:
# Interactive test input to test the model
# test_input = input("Enter a message to classify as Ham or Spam (PLEASE INPUT PREPROCESSED :) ): ")
test_input = ["free money education"]

# Transform
test_tfidf = tfidf.transform(test_input)

# Make prediction
prediction = svm_clf.predict(test_tfidf)
probability = svm_clf.decision_function(test_tfidf)

label = "Spam" if prediction[0] == 1 else "Ham"
print(f"\nPrediction: {label}")
print(f"Confidence Score: {abs(probability[0]):.4f}")


Prediction: Spam
Confidence Score: 0.2023


In [56]:
from collections import Counter
import numpy as np
def scratch_tf_idf(documents, vocab=None, smooth_idf=True, max_features=5000):
    """
    TF-IDF Formula:
    - TF(t, d) = (number of times term t appears in document d) / (total number of terms in document d)
    - IDF(t) = log((N + 1) / (DF(t) + 1)) + 1  (with smoothing, where N is total documents, DF(t) is documents containing t)
      or IDF(t) = log(N / DF(t))  (without smoothing)
    - TF-IDF(t, d) = TF(t, d) * IDF(t)
    """
    # Tokenize documents into lists of words
    tokenized = [doc.split() for doc in documents]

    # If no vocab provided, build from document tokens
    if vocab is None:
        # Count occurrences of each word across all documents
        word_counts = Counter(word for tokens in tokenized for word in tokens)

        # Limit to max_features most common words if vocabulary too large
        if len(word_counts) > max_features:
            word_counts = dict(word_counts.most_common(max_features))
            
        # Sort vocabulary for consistent ordering
        vocab = sorted(word_counts)

    # Create mapping from word to index in vocabulary
    vocab_index = {word: i for i, word in enumerate(vocab)}
    n_docs = len(tokenized)
    n_terms = len(vocab)

    # Initialize TF matrix: rows are documents, columns are terms
    tf = np.zeros((n_docs, n_terms), dtype=np.float32)

    # Compute Term Frequency (TF) for each document
    for i, tokens in enumerate(tokenized):
        if not tokens:
            continue
        # Get indices of tokens that are in vocabulary (drop out-of-vocabulary words)
        indices = np.fromiter(
            (vocab_index[w] for w in tokens if w in vocab_index),
            dtype=np.int32
        )
        if indices.size == 0:
            continue
        # Count occurrences of each term in this document using bincount
        # Normalize by total tokens in document to get TF
        tf[i] = np.bincount(indices, minlength=n_terms) / len(tokens)

    # Document Frequency (DF): number of documents containing each term
    df = np.count_nonzero(tf, axis=0)  # Shape: (n_terms,)

    # Compute Inverse Document Frequency (IDF)
    if smooth_idf:
        # Smoothed IDF: log((N+1)/(DF+1)) + 1
        idf = np.log((n_docs + 1) / (df + 1)) + 1
    else:
        # Standard IDF: log(N / DF), avoid log(0) by max(DF, 1)
        idf = np.log(n_docs / np.maximum(df, 1))

    # Compute TF-IDF: element-wise multiplication (broadcasting)
    tfidf = tf * idf  # Shape: (n_docs, n_terms)

    return tfidf, vocab

In [57]:
# Fit on train (build vocab)
X_train_tfidf, vocab = scratch_tf_idf(
    X_train.tolist(),
    vocab=None,          # learn vocab from training data
    smooth_idf=True,
    max_features=10000   # match your sklearn setup
)

# Transform test
X_test_tfidf, _ = scratch_tf_idf(
    X_test.tolist(),
    vocab=vocab,
    smooth_idf=True,
    max_features=10000
)

print(f"Train shape: {X_train_tfidf.shape}")
print(f"Test shape:  {X_test_tfidf.shape}")

Train shape: (4836, 10000)
Test shape:  (1209, 10000)


In [58]:
# SVM
svm_clf = SVC(kernel='linear', C=1.0, random_state=42)
svm_clf.fit(X_train_tfidf, y_train)
svm_pred = svm_clf.predict(X_test_tfidf)

In [59]:
print(classification_report(y_test, svm_pred, target_names=['Ham', 'Spam']))
print(f"Accuracy: {accuracy_score(y_test, svm_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, svm_pred):.4f}")

              precision    recall  f1-score   support

         Ham       0.97      0.98      0.98       830
        Spam       0.96      0.93      0.95       379

    accuracy                           0.97      1209
   macro avg       0.97      0.96      0.96      1209
weighted avg       0.97      0.97      0.97      1209

Accuracy: 0.9669
F1 Score: 0.9462
